# Cross-noise generalization — QTDB × NSTDB BW/EM/MA

Notebook đánh giá chéo toàn bộ ma trận **model huấn luyện theo BW/EM/MA × dữ liệu kiểm tra BW/EM/MA**. Hai protocol được giữ riêng và ghép đúng protocol giữa model và test để bảo toàn quy tắc tách kênh/nửa thời gian của noise.

Mục tiêu chính là đo hiệu năng đúng miền, hiệu năng trên hai loại nhiễu chưa gặp, mức suy giảm OOD và trường hợp OOD xấu nhất. Mọi model dùng cùng QTDB segments, cùng noisy input, cùng mức `r`, và cùng trạng thái RNG cho từng test noise/protocol.

**Kaggle:** bật GPU + Internet, dùng **Add Input** để gắn dataset chứa sáu file `best_<bw|em|ma>_protocol<1|2>_norm_loss_v2.pth`, sau đó chạy **Run all**. Mặc định là full test, DDIM 50 bước, `eta=0`, 1-shot. Nếu hiện chỉ có đúng ba checkpoint của một protocol, đổi `PROTOCOLS = (1,)` hoặc `(2,)` cho phù hợp.


## 1. Cài thư viện


In [ ]:
%pip -q install 'wfdb>=4.1,<5' 'scipy>=1.10' 'pandas>=2.0' 'matplotlib>=3.7' 'pyyaml>=6.0'


## 2. Cấu hình ma trận đánh giá chéo


In [ ]:
import hashlib
import json
import platform
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import signal
import torch
from tqdm.auto import tqdm
import wfdb
from IPython.display import display

SEED = 1234
MODEL_ARTIFACTS = ('BW', 'EM', 'MA')
TEST_ARTIFACTS = ('BW', 'EM', 'MA')
PROTOCOLS = (1, 2)
# Start with 1-shot. Add 3, 5, 10 only after the full 3x3 smoke test passes.
K_VALUES = (1,)
BATCH_SIZE = 50
DDIM_STEPS = 50
DDIM_ETA = 0.0

# Attach one Kaggle dataset containing the six protocol checkpoints.
CHECKPOINT_ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working/qtdb_cross_noise_generalization')
OUT.mkdir(parents=True, exist_ok=True)

TEST_RECORDS = (
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853',
    'sel16420', 'sel16795', 'sele0106', 'sele0121', 'sel32',
    'sel49', 'sel14046', 'sel15814',
)
TARGET_FS = 360
LENGTH = 512
MAX_UNPADDED = 496
INSERT_OFFSET = 16
PREPROCESSING_VERSION = 'qtdb_pu1_p_and_beat_40ms_reflect100ms_360hz_endpoints_pad16_v2'
MODEL_NORMALIZATION = 'per_segment_clean_maxabs_shared_with_noisy_v1'
LOSS_VERSION = 'l1_mean_relative_stft_snr_weighted_continuous_alpha_v1'
CHECKPOINT_TAG = 'norm_loss_v2'
PHYSIONET_QT = 'qtdb/1.0.0'
PHYSIONET_NST = 'nstdb/1.0.0'
VALID_ARTIFACTS = {'BW': 'bw', 'EM': 'em', 'MA': 'ma'}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable GPU in Kaggle: Settings -> Accelerator -> GPU.')

# Use small values only for a smoke test. Keep None for the official run.
MAX_TEST_RECORDS = None
MAX_TEST_SEGMENTS = None

CONFIG = {
    'train': {'feats': 80, 'batch_size': 96, 'epochs': 100, 'lr': 1e-4,
              'lambda_time': 1.0, 'lambda_freq': 0.1},
    'diffusion': {'beta_start': 1e-4, 'beta_end': 0.5,
                  'num_steps': 50, 'schedule': 'quad'},
    'model_normalization': MODEL_NORMALIZATION,
    'loss_version': LOSS_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
}

if not set(MODEL_ARTIFACTS).issubset(VALID_ARTIFACTS):
    raise ValueError('MODEL_ARTIFACTS contains an unknown artifact')
if not set(TEST_ARTIFACTS).issubset(VALID_ARTIFACTS):
    raise ValueError('TEST_ARTIFACTS contains an unknown artifact')
if not set(PROTOCOLS).issubset({1, 2}):
    raise ValueError('PROTOCOLS must contain only 1 and/or 2')

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('GPU:', torch.cuda.get_device_name(0))
print('Checkpoint root:', CHECKPOINT_ROOT, '| output:', OUT)


## 3. Kiến trúc HNF U-Net + FiLM + attention

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math
import torch
import torch.nn as nn

class HNFBlock(nn.Module):
    """
    Half Normalized Filter block.
    Multi-scale convs → concat → conv → half‑instance norm → residual.
    """
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k,
                      padding=k//2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        # Half‑instance norm: chia channel làm 2 nửa
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        multi_out = [conv(x) for conv in self.multi_convs]
        out = torch.cat(multi_out, dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out_norm = self.half_inst_norm(out[:, :half, :])
        out = torch.cat([out_norm, out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)
import torch
import torch.nn as nn
import math

class BridgeBlock(nn.Module):
    """FiLM‑based conditioning on noise level (sqrt(alpha_bar))."""
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(
            nn.Linear(emb_dim, features * 2),
            nn.SiLU()
        )

    def forward(self, x, alpha_bar):
        # alpha_bar: (B,) or scalar
        # Ép phẳng alpha_bar về dạng 1D (Batch,) để tránh bị dư chiều -- CỰC QUAN TRỌNG
        alpha_bar = alpha_bar.view(-1)
        emb = self.sinusoidal_embedding(alpha_bar)
        scale_shift = self.film(emb)   # (B, 2*features)
        scale, shift = scale_shift.chunk(2, dim=1)
        scale = scale.unsqueeze(-1)
        shift = shift.unsqueeze(-1)
        return x * (1 + scale) + shift

    def sinusoidal_embedding(self, x):
        if x.dim() == 0:
            x = x.unsqueeze(0)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb
import torch
import torch.nn as nn

class SelfAttention1D(nn.Module):
    """1D Self‑Attention (multi‑head) dùng tại bottleneck."""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels, "channels phải chia hết cho num_heads"

        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, L = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, L)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        # q, k, v: (B, num_heads, head_dim, L)
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        out = out.reshape(B, C, L)
        return self.proj(out)
import torch
import torch.nn as nn

class UNet1D(nn.Module):
    """
    1D U‑Net với 4 levels, skip connections, HNF blocks,
    Bridge FiLM (timestep), và Self‑Attention tại bottleneck.
    """
    def __init__(self, in_channels=24, base_channels=64, emb_dim=128, out_channels=12):
        """
        in_channels = 24 vì concat x_t (12 channels) và condition (12 channels)
        """
        super().__init__()
        # ---------- ENCODER ----------
        # Level 1 (input -> 64)
        self.enc1 = HNFBlock(in_channels, base_channels)
        self.bridge1 = BridgeBlock(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels*2, kernel_size=4, stride=2, padding=1)

        # Level 2 (64 -> 128)
        self.enc2 = HNFBlock(base_channels*2, base_channels*2)
        self.bridge2 = BridgeBlock(base_channels*2, emb_dim)
        self.down2 = nn.Conv1d(base_channels*2, base_channels*4, kernel_size=4, stride=2, padding=1)

        # Level 3 (128 -> 256)
        self.enc3 = HNFBlock(base_channels*4, base_channels*4)
        self.bridge3 = BridgeBlock(base_channels*4, emb_dim)
        self.down3 = nn.Conv1d(base_channels*4, base_channels*8, kernel_size=4, stride=2, padding=1)

        # Level 4 (256 -> 512) – bottleneck
        self.enc4 = HNFBlock(base_channels*8, base_channels*8)
        self.bridge4 = BridgeBlock(base_channels*8, emb_dim)
        # Self‑Attention tại bottleneck
        self.attn = SelfAttention1D(base_channels*8)

        # ---------- DECODER ----------
        # Level 4 -> 3
        self.up4 = nn.ConvTranspose1d(base_channels*8, base_channels*4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlock(base_channels*8, base_channels*4)   # concat với skip từ enc3

        # Level 3 -> 2
        self.up3 = nn.ConvTranspose1d(base_channels*4, base_channels*2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlock(base_channels*4, base_channels*2)   # concat với skip từ enc2

        # Level 2 -> 1
        self.up2 = nn.ConvTranspose1d(base_channels*2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlock(base_channels*2, base_channels)     # concat với skip từ enc1

        # Output
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        """
        x: latent (x_t), cond: noisy observation, noise_scale: sqrt(alpha_bar)
        """
        # Concatenate x and cond along channel dimension
        inp = torch.cat([x, cond], dim=1)   # (B, 2, L)

        # ----- Encoder -----
        e1 = self.enc1(inp)
        e1 = self.bridge1(e1, noise_scale)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        e2 = self.bridge2(e2, noise_scale)
        d2 = self.down2(e2)

        e3 = self.enc3(d2)
        e3 = self.bridge3(e3, noise_scale)
        d3 = self.down3(e3)

        e4 = self.enc4(d3)
        e4 = self.bridge4(e4, noise_scale)
        # Bottleneck self‑attention
        e4 = self.attn(e4)

        # ----- Decoder (with skip connections) -----
        u4 = self.up4(e4)
        u4 = torch.cat([u4, e3], dim=1)   # skip từ enc3
        d4 = self.dec4(u4)

        u3 = self.up3(d4)
        u3 = torch.cat([u3, e2], dim=1)   # skip từ enc2
        d3 = self.dec3(u3)

        u2 = self.up2(d3)
        u2 = torch.cat([u2, e1], dim=1)   # skip từ enc1
        d2 = self.dec2(u2)

        out = self.final(d2)
        return out

## 4. Conditional DDPM/DDIM

In [ ]:
from functools import partial
from inspect import isfunction
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from inspect import isfunction

# ----- Bắt buộc phải có torchaudio hoặc dùng torch.stft -----
# Nếu không có torchaudio, ta dùng torch.stft nguyên bản
def stft_loss(pred, target, n_fft=128, hop_length=64):
    """
    Tính MSE loss trên phổ STFT giữa pred và target.
    """
    # (B, C, L) -> (B*C, L)
    B, C, L = pred.shape
    pred = pred.view(-1, L)
    target = target.view(-1, L)
    
    # STFT: returns (B, freq_bins, time_frames, 2)
    spec_pred = torch.stft(pred, n_fft=n_fft, hop_length=hop_length, 
                           return_complex=True)
    spec_target = torch.stft(target, n_fft=n_fft, hop_length=hop_length,
                             return_complex=True)
    
    # Lấy magnitude (độ lớn)
    mag_pred = torch.abs(spec_pred)
    mag_target = torch.abs(spec_target)
    
    return F.mse_loss(mag_pred, mag_target)


def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d


class DDPM(nn.Module):
    def __init__(self, base_model, config, device, conditional=True):
        super().__init__()
        self.device = device
        self.model = base_model
        self.config = config
        self.conditional = conditional
        
        # ----------------------
        # 👉 THÊM: Loss weights
        # ----------------------
        self.lambda_time = config['train'].get('lambda_time', 1.0)
        self.lambda_freq = config['train'].get('lambda_freq', 0.1)  # cân bằng
        
        self.loss_func_l1 = nn.L1Loss(reduction='sum').to(device)
        
        config_diff = config["diffusion"]
        self.num_steps = config_diff["num_steps"]
        self.set_new_noise_schedule(config_diff, device)
        
    def make_beta_schedule(self, schedule='linear', n_timesteps=1000, start=1e-5, end=1e-2):
        if schedule == 'linear':
            betas = torch.linspace(start, end, n_timesteps)
        elif schedule == "quad":
            betas = torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
        elif schedule == "sigmoid":
            betas = torch.linspace(-6, 6, n_timesteps)
            betas = torch.sigmoid(betas) * (end - start) + start
        return betas
    
    def set_new_noise_schedule(self, config_diff, device):
        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)
        
        betas = self.make_beta_schedule(
            schedule=config_diff["schedule"], 
            n_timesteps=config_diff["num_steps"],
            start=config_diff["beta_start"], 
            end=config_diff["beta_end"]
        )
        betas = betas.detach().cpu().numpy() if isinstance(betas, torch.Tensor) else betas
        
        alphas = 1. - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod_prev = np.sqrt(np.append(1., alphas_cumprod))
        
        self.register_buffer('betas', to_torch(betas))
        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))
        self.register_buffer('alphas_cumprod_prev', to_torch(alphas_cumprod_prev))
        self.register_buffer('sqrt_alphas_cumprod', to_torch(np.sqrt(alphas_cumprod)))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', to_torch(np.sqrt(1. - alphas_cumprod)))
        self.register_buffer('log_one_minus_alphas_cumprod', to_torch(np.log(1. - alphas_cumprod)))
        self.register_buffer('sqrt_recip_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod)))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod - 1)))
        
        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        self.register_buffer('posterior_variance', to_torch(posterior_variance))
        self.register_buffer('posterior_log_variance_clipped', 
                             to_torch(np.log(np.maximum(posterior_variance, 1e-20))))
        self.register_buffer('posterior_mean_coef1', 
                             to_torch(betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))
        self.register_buffer('posterior_mean_coef2', 
                             to_torch((1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))
        
    def predict_start_from_noise(self, x_t, t, noise):
        # Nắn lại chiều tensor hệ số thành (Batch, 1, 1) để nhân đúng với tensor tín hiệu 3D
        coef1 = self.sqrt_recip_alphas_cumprod[t].view(-1, 1, 1)
        coef2 = self.sqrt_recipm1_alphas_cumprod[t].view(-1, 1, 1)
        
        return coef1 * x_t - coef2 * noise
    
    def q_posterior(self, x_start, x_t, t):
        posterior_mean = self.posterior_mean_coef1[t] * x_start + \
                         self.posterior_mean_coef2[t] * x_t
        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]
        return posterior_mean, posterior_log_variance_clipped
    
    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):
        batch_size = x.shape[0]
        noise_level = torch.FloatTensor(
            [self.sqrt_alphas_cumprod_prev[t+1]]
        ).repeat(batch_size, 1).to(x.device)
        if condition_x is not None:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, condition_x, noise_level))
        else:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, noise_level))
        if clip_denoised:
            x_recon.clamp_(-1., 1.)
        model_mean, posterior_log_variance = self.q_posterior(
            x_start=x_recon, x_t=x, t=t)
        return model_mean, posterior_log_variance
    
    @torch.no_grad()
    def p_sample(self, x, t, clip_denoised=False, condition_x=None):
        model_mean, model_log_variance = self.p_mean_variance(
            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        return model_mean + noise * (0.5 * model_log_variance).exp()

    @torch.no_grad()
    def p_sample_loop(self, x_in, continous=False):
        device = self.betas.device
        sample_inter = (1 | (self.num_steps//10))
        if not self.conditional:
            shape = x_in
            cur_x = torch.randn(shape, device=device)
            ret_x = cur_x
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i)
                if i % sample_inter == 0:
                    ret_x = torch.cat([ret_x, cur_x], dim=0)
        else:
            x = x_in
            shape = x.shape
            cur_x = torch.randn(shape, device=device)
            ret_x = [cur_x]
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i, condition_x=x)
                if i % sample_inter == 0:
                    ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]    
    
    @torch.no_grad()
    def sample(self, batch_size=1, shape=[1, 512], continous=False):
        return self.p_sample_loop((batch_size, shape[0], shape[1]), continous)
    
    # ==========================================================
    # 👉 DDIM SAMPLER (đã có sẵn trong source gốc)
    # ==========================================================
    @torch.no_grad()
    def ddim_sample_loop(self, x_in, ddim_timesteps=50, ddim_eta=0.0, num_shots=1):
        device = self.betas.device
        T = self.num_steps
        S = ddim_timesteps
        tau = [int(np.floor((T / (S ** 2)) * (i ** 2))) for i in range(S + 1)]
        
        alphas_cumprod_with_1 = torch.cat([
            torch.tensor([1.0], dtype=self.alphas_cumprod.dtype, device=device),
            self.alphas_cumprod
        ]).float()
        
        out_accum = 0.0
        for shot in range(num_shots):
            shape = x_in.shape
            x = torch.randn(shape, device=device)
            for i in reversed(range(1, S + 1)):
                t = tau[i]
                t_prev = tau[i - 1]
                noise_level = torch.FloatTensor(
                    [self.sqrt_alphas_cumprod_prev[t]]
                ).repeat(x.shape[0], 1).to(device)
                if not self.conditional:
                    eps = self.model(x, noise_level)
                else:
                    eps = self.model(x, x_in, noise_level)
                    
                a_t = alphas_cumprod_with_1[t]
                a_t_prev = alphas_cumprod_with_1[t_prev]
                x0_pred = (x - torch.sqrt(1. - a_t) * eps) / torch.sqrt(a_t)
                if t_prev == 0:
                    sigma_t = 0.0
                else:
                    sigma_t = ddim_eta * torch.sqrt((1. - a_t_prev) / (1. - a_t)) * \
                              torch.sqrt(torch.clamp(1. - a_t / a_t_prev, min=0.0))
                direction = torch.sqrt(torch.clamp(1. - a_t_prev - sigma_t**2, min=0.0)) * eps
                noise = sigma_t * torch.randn_like(x) if (ddim_eta > 0 and t_prev > 0) else 0.0
                x = torch.sqrt(a_t_prev) * x0_pred + direction + noise
            out_accum += x
        return out_accum / num_shots

    @torch.no_grad()
    def denoising(self, x_in, continous=False, use_ddim=False, ddim_steps=50, ddim_eta=0.0, num_shots=1):
        if use_ddim:
            return self.ddim_sample_loop(x_in, ddim_timesteps=ddim_steps, ddim_eta=ddim_eta, num_shots=num_shots)
        else:
            if num_shots > 1:
                out_accum = 0.0
                for _ in range(num_shots):
                    out_accum += self.p_sample_loop(x_in, continous)
                return out_accum / num_shots
            return self.p_sample_loop(x_in, continous)
    
    def q_sample_loop(self, x_start, continous=False):
        sample_inter = (1 | (self.num_steps//10))
        ret_x = [x_start]
        cur_x = x_start
        for t in range(1, self.num_steps+1):
            B,C,L = cur_x.shape
            continuous_sqrt_alpha_cumprod = torch.FloatTensor(
                np.random.uniform(
                    self.sqrt_alphas_cumprod_prev[t-1],
                    self.sqrt_alphas_cumprod_prev[t],
                    size=B
                )
            ).to(cur_x.device)
            continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
            noise = torch.randn_like(cur_x)
            cur_x = self.q_sample(
                x_start=cur_x, 
                continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
                noise=noise
            )
            if t % sample_inter == 0:
                ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]
    
    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise=None):
        noise = default(noise, lambda: torch.randn_like(x_start))
        return continuous_sqrt_alpha_cumprod * x_start + \
               (1 - continuous_sqrt_alpha_cumprod**2).sqrt() * noise
    
    # ==========================================================
    # 👉 HÀM LOSS CHÍNH (ĐÃ THÊM STFT)
    # ==========================================================
    def p_losses(self, x_in, y_in, noise=None):
        """
        Multi‑domain Loss:
        - L1 trên noise (miền thời gian)
        - STFT Loss trên tín hiệu dự đoán (miền tần số)
        """
        x_start = x_in
        B, C, L = x_start.shape

        # Lấy ngẫu nhiên các giá trị t khác nhau cho từng mẫu trong Batch, trực tiếp trên thiết bị (GPU)
        t = torch.randint(0, self.num_steps, (B,), device=x_start.device).long()
        # 1. Chuyển mảng NumPy sang PyTorch Tensor và đẩy lên cùng GPU với t
        sqrt_alphas_prev_tensor = torch.tensor(self.sqrt_alphas_cumprod_prev, dtype=torch.float32, device=t.device)
        # 2. Lấy giới hạn dưới và trên. 
        # (Lưu ý: Do t giờ chạy từ 0 đến num_steps-1, ta dùng chỉ số t và t+1 để thay thế cho t-1 và t nhằm tránh lỗi âm)
        lower = sqrt_alphas_prev_tensor[t]
        upper = sqrt_alphas_prev_tensor[t+1]
        # 3. Sinh số ngẫu nhiên uniform hoàn toàn bằng PyTorch trực tiếp trên GPU
        continuous_sqrt_alpha_cumprod = lower + torch.rand(B, device=t.device) * (upper - lower)
        continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
        noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(
            x_start=x_start, 
            continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
            noise=noise
        )

        # Dự đoán noise
        if not self.conditional:
            x_recon = self.model(x_noisy, continuous_sqrt_alpha_cumprod)
        else:
            x_recon = self.model(x_noisy, y_in, continuous_sqrt_alpha_cumprod)

        # ---------- 1. L1 Loss (miền thời gian) ----------
        loss_l1 = self.loss_func_l1(noise, x_recon)  # (B*C*L)

        # ---------- 2. STFT Loss (miền tần số) ----------
        # Dự đoán x0 từ noise đã dự đoán
        x0_pred = self.predict_start_from_noise(x_noisy, t, x_recon)
        
        # Tính STFT loss giữa x0_pred và x_start (clean gốc)
        loss_stft = stft_loss(x0_pred, x_start, n_fft=128, hop_length=64)
        
        # ---------- 3. Tổng hợp ----------
        total_loss = self.lambda_time * loss_l1 + self.lambda_freq * loss_stft
        return total_loss
    
    def forward(self, x, y, *args, **kwargs):
        return self.p_losses(x, y, *args, **kwargs)


class EMA(object):
    def __init__(self, mu=0.999):
        self.mu = mu
        self.shadow = {}

    def register(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name].data = (1. - self.mu) * param.data + self.mu * self.shadow[name].data

    def ema(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name].data)

    def ema_copy(self, module):
        module_copy = type(module)(module.config).to(module.config.device)
        module_copy.load_state_dict(module.state_dict())
        self.ema(module_copy)
        return module_copy

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, state_dict):
        self.shadow = state_dict

class ResearchLossDDPM(DDPM):
    """Same DDPM/UNet architecture, with a scale-stable multi-domain objective."""

    def loss_components(self, x_in, y_in, noise=None):
        x_start = x_in
        batch = x_start.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=x_start.device)
        boundaries = torch.as_tensor(
            self.sqrt_alphas_cumprod_prev, dtype=x_start.dtype, device=x_start.device
        )
        lower, upper = boundaries[timestep], boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=x_start.device) * (upper - lower)
        continuous_2d = continuous.view(batch, 1)
        continuous_3d = continuous.view(batch, 1, 1)
        gaussian_noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(x_start, continuous_3d, gaussian_noise)
        predicted_noise = self.model(x_noisy, y_in, continuous_2d)

        # Mean keeps the time loss independent of batch size and 512-sample length.
        loss_time = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')

        # Reconstruct x0 with the exact continuous alpha used to create x_t.
        residual_scale = torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0))
        x0_prediction = (x_noisy - residual_scale * predicted_noise) / continuous_3d.clamp_min(1e-6)

        n_fft, hop_length = 128, 64
        window = torch.hann_window(n_fft, dtype=x_start.dtype, device=x_start.device)
        pred_flat = x0_prediction.reshape(-1, x0_prediction.shape[-1])
        target_flat = x_start.reshape(-1, x_start.shape[-1])
        pred_mag = torch.abs(torch.stft(pred_flat, n_fft=n_fft, hop_length=hop_length,
                                         window=window, return_complex=True))
        target_mag = torch.abs(torch.stft(target_flat, n_fft=n_fft, hop_length=hop_length,
                                           window=window, return_complex=True))
        pred_mag = pred_mag.reshape(batch, -1)
        target_mag = target_mag.reshape(batch, -1)
        spectral_error = torch.mean((pred_mag - target_mag) ** 2, dim=1)
        target_power = torch.mean(target_mag ** 2, dim=1).clamp_min(1e-6)
        relative_frequency = spectral_error / target_power

        # x0 is ill-conditioned at high-noise timesteps, so weight STFT by diffusion SNR.
        alpha_bar = continuous ** 2
        diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
        frequency_weight = torch.clamp(diffusion_snr, max=1.0)
        loss_frequency = torch.mean(relative_frequency * frequency_weight)
        total = self.lambda_time * loss_time + self.lambda_freq * loss_frequency
        return {'total': total, 'time': loss_time, 'frequency': loss_frequency,
                'frequency_unweighted': relative_frequency.mean(),
                'frequency_weight': frequency_weight.mean()}

    def p_losses(self, x_in, y_in, noise=None):
        return self.loss_components(x_in, y_in, noise)['total']


## 5. Nạp và kiểm tra checkpoint EMA theo miền train


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


MODELS, CHECKPOINT_INFO = {}, []
for model_artifact in MODEL_ARTIFACTS:
    for protocol in PROTOCOLS:
        filename = f'best_{model_artifact.lower()}_protocol{protocol}_{CHECKPOINT_TAG}.pth'
        matches = sorted(CHECKPOINT_ROOT.rglob(filename))
        if len(matches) != 1:
            raise FileNotFoundError(f'Need exactly one {filename}; found {matches}')
        path = matches[0]
        payload = torch.load(path, map_location='cpu', weights_only=False)
        if payload.get('artifact') != model_artifact or int(payload.get('protocol', -1)) != protocol:
            raise RuntimeError(f'Checkpoint identity mismatch: {path}')
        if payload.get('config') != CONFIG:
            raise RuntimeError(f'Checkpoint config mismatch: {path}')
        if 'ema_model_state_dict' not in payload:
            raise RuntimeError(f'EMA inference weights missing: {path}')
        model = ResearchLossDDPM(UNet1D(2, 80, 128, 1), CONFIG, DEVICE).to(DEVICE)
        model.load_state_dict(payload['ema_model_state_dict'], strict=True)
        model.eval()
        probe = torch.zeros(1, 1, LENGTH, device=DEVICE)
        with torch.inference_mode():
            probe_output = model.model(probe, probe, torch.ones(1, 1, device=DEVICE))
        assert probe_output.shape == probe.shape and torch.isfinite(probe_output).all()
        MODELS[(model_artifact, protocol)] = model
        CHECKPOINT_INFO.append({
            'model_artifact': model_artifact,
            'protocol': protocol,
            'path': str(path),
            'sha256': sha256(path),
            'selected_epoch': int(payload['epoch']) + 1,
            'best_validation_loss': float(payload['best_validation_loss']),
            'parameter_count': sum(parameter.numel() for parameter in model.parameters()),
        })

checkpoint_manifest = pd.DataFrame(CHECKPOINT_INFO)
checkpoint_manifest.to_csv(OUT / 'checkpoint_manifest.csv', index=False)
display(checkpoint_manifest)


## 6. Hàm tạo QTDB clean segments

In [ ]:
BEAT_SYMBOLS = set('NLRBAaJSVrFejnE/fQ?')

def p_onsets(annotation):
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample, dtype=np.int64)
    return np.asarray([int(samples[i]) for i, symbol in enumerate(symbols)
                       if symbol == '(' and 'p' in symbols[i + 1:i + 4]], dtype=np.int64)

def r_locations(annotation):
    return np.asarray([int(sample) for sample, symbol in zip(annotation.sample, annotation.symbol)
                       if symbol in BEAT_SYMBOLS], dtype=np.int64)

def resample_reflect(segment, original_fs):
    pad = min(max(1, int(round(0.1 * original_fs))), len(segment) - 1)
    padded = np.pad(segment, (pad, pad), mode='reflect')
    new_length = int(round(len(padded) * TARGET_FS / original_fs))
    resampled = signal.resample(padded, new_length).astype(np.float32)
    target_pad = int(round(pad * TARGET_FS / original_fs))
    expected = int(round(len(segment) * TARGET_FS / original_fs))
    return resampled[target_pad:target_pad + expected]

def clean_segments_for_record(name):
    record = wfdb.rdrecord(name, pn_dir=PHYSIONET_QT, channels=[0])
    # QTDB does not provide .atr for every record (e.g. sel30).
    # .pu1 provides P boundaries and beat fiducials throughout each record.
    pu = wfdb.rdann(name, 'pu1', pn_dir=PHYSIONET_QT)
    fs = float(record.fs)
    ecg = record.p_signal[:, 0].astype(np.float32)
    starts = p_onsets(pu) - int(round(0.04 * fs))
    starts = starts[(starts >= 0) & (starts < len(ecg))]
    peaks = r_locations(pu)
    beats, rows = [], []
    audit = {'record_id': name, 'p_onsets': len(starts),
             'candidate_intervals': max(0, len(starts) - 1),
             'rejected_short_interval': 0, 'rejected_multiple_r': 0,
             'rejected_length': 0, 'rejected_nonfinite_or_flat': 0}
    resampled_lengths = []
    for segment_index, (start, end) in enumerate(zip(starts[:-1], starts[1:])):
        if end <= start + 2:
            audit['rejected_short_interval'] += 1
            continue
        r_count = int(np.count_nonzero((peaks >= start) & (peaks < end)))
        if r_count > 1:
            audit['rejected_multiple_r'] += 1
            continue
        segment = resample_reflect(ecg[start:end], fs)
        resampled_lengths.append(len(segment))
        if not 2 <= len(segment) <= MAX_UNPADDED:
            audit['rejected_length'] += 1
            continue
        segment = segment - 0.5 * (segment[0] + segment[-1])
        clean = np.zeros(LENGTH, dtype=np.float32)
        clean[INSERT_OFFSET:INSERT_OFFSET + len(segment)] = segment
        if not np.all(np.isfinite(clean)) or np.ptp(clean) <= 0:
            audit['rejected_nonfinite_or_flat'] += 1
            continue
        beats.append(clean)
        rows.append({'sample_id': f'{name}:{segment_index}', 'patient_id': name,
                     'record_id': name, 'clean_segment_id': f'{name}:{segment_index}',
                     'start_sample': int(start), 'end_sample': int(end),
                     'unpadded_length': len(segment), 'r_count': r_count,
                     'annotation_source': 'pu1'})
    audit['min_resampled_length'] = min(resampled_lengths) if resampled_lengths else np.nan
    audit['max_resampled_length'] = max(resampled_lengths) if resampled_lengths else np.nan
    audit['accepted_segments'] = len(beats)
    return beats, rows, audit



## 7. Tạo QTDB test set và tải BW/EM/MA


In [ ]:
selected_records = TEST_RECORDS[:MAX_TEST_RECORDS] if MAX_TEST_RECORDS else TEST_RECORDS
all_beats, rows, audits = [], [], []
for record_name in tqdm(selected_records, desc='QTDB test records'):
    record_beats, record_rows, audit = clean_segments_for_record(record_name)
    all_beats.extend(record_beats)
    rows.extend(record_rows)
    audits.append(audit)
    if MAX_TEST_SEGMENTS and len(all_beats) >= MAX_TEST_SEGMENTS:
        all_beats = all_beats[:MAX_TEST_SEGMENTS]
        rows = rows[:MAX_TEST_SEGMENTS]
        break
if not all_beats:
    raise RuntimeError('No QTDB test segments were produced')
CLEAN = np.stack(all_beats).astype(np.float32)
CLEAN_META = pd.DataFrame(rows)
pd.DataFrame(audits).to_csv(OUT / 'test_record_audit.csv', index=False)
np.save(OUT / 'clean_test.npy', CLEAN)
assert CLEAN.shape == (len(CLEAN_META), LENGTH) and CLEAN_META.sample_id.is_unique
print('Clean QTDB test:', CLEAN.shape)

NOISE, MIDPOINT = {}, {}
for artifact, record_name in VALID_ARTIFACTS.items():
    record = wfdb.rdrecord(record_name, pn_dir=PHYSIONET_NST)
    values = record.p_signal.astype(np.float32)
    if int(record.fs) != TARGET_FS:
        values = signal.resample(
            values, int(round(len(values) * TARGET_FS / float(record.fs))), axis=0
        ).astype(np.float32)
    assert values.ndim == 2 and values.shape[1] >= 2 and np.isfinite(values).all()
    NOISE[artifact] = values
    MIDPOINT[artifact] = len(values) // 2


## 8. Tạo noisy input dùng chung cho mọi model


In [ ]:
def make_test_pairs(test_artifact, protocol):
    # P1 trains on first-half channel 1 and tests on second-half channel 2.
    # P2 trains on first-half channel 2 and tests on second-half channel 1.
    channel = 1 if protocol == 1 else 0
    midpoint = MIDPOINT[test_artifact]
    noise = NOISE[test_artifact][midpoint:, channel]
    patch_count = len(noise) // LENGTH
    if patch_count == 0:
        raise RuntimeError(f'{test_artifact} test half is shorter than one patch')
    # Reuse the same r vector for all artifacts/protocols for paired comparisons.
    rng = np.random.RandomState(SEED)
    levels = rng.randint(20, 200, size=len(CLEAN)).astype(np.float32) / 100.0
    noisy = np.empty_like(CLEAN)
    rows = []
    for index, (clean, level) in enumerate(zip(CLEAN, levels)):
        relative_start = (index % patch_count) * LENGTH
        patch = noise[relative_start:relative_start + LENGTH]
        clean_range, noise_range = float(np.ptp(clean)), float(np.ptp(patch))
        if clean_range <= 0 or noise_range <= 0:
            raise ValueError(f'Zero range: {test_artifact}/P{protocol}/sample{index}')
        noisy[index] = clean + float(level) * clean_range / noise_range * patch
        row = CLEAN_META.iloc[index].to_dict()
        row.update({
            'test_artifact': test_artifact,
            'protocol': protocol,
            'split': 'test',
            'noise_source_id': f'nstdb/1.0.0/{VALID_ARTIFACTS[test_artifact]}:second_half',
            'noise_channel': channel + 1,
            'noise_start': midpoint + relative_start,
            'r': float(level),
            'seed': SEED,
            'sampling_rate': TARGET_FS,
            'preprocessing_version': PREPROCESSING_VERSION,
        })
        rows.append(row)
    return noisy, pd.DataFrame(rows)


NOISY, MANIFEST = {}, {}
for test_artifact in TEST_ARTIFACTS:
    for protocol in PROTOCOLS:
        key = (test_artifact, protocol)
        NOISY[key], MANIFEST[key] = make_test_pairs(test_artifact, protocol)

manifest = pd.concat(MANIFEST.values(), ignore_index=True)
assert len(manifest) == len(CLEAN) * len(TEST_ARTIFACTS) * len(PROTOCOLS)
assert not manifest.duplicated(['test_artifact', 'protocol', 'sample_id']).any()
manifest.to_csv(OUT / 'test_pair_manifest.csv', index=False)
display(manifest.groupby(['test_artifact', 'protocol', 'noise_channel']).size().rename('N'))


## 9. DDIM inference cho ma trận model × test noise


In [ ]:
def synchronize():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def compute_metric_frame(clean, noisy, prediction, metadata, model_artifact,
                         test_artifact, protocol, k, elapsed_ms, output_start,
                         checkpoint_hash):
    clean, noisy, prediction = [np.asarray(values, dtype=np.float64)
                                for values in (clean, noisy, prediction)]
    error = prediction - clean
    numerator = np.sum(error ** 2, axis=1)
    signal_power = np.sum(clean ** 2, axis=1)
    input_power = np.sum((noisy - clean) ** 2, axis=1)
    repo_denominator = np.sum((prediction - np.mean(clean)) ** 2, axis=1)
    standard_denominator = np.sum((clean - clean.mean(axis=1, keepdims=True)) ** 2, axis=1)
    cosine_denominator = np.linalg.norm(clean, axis=1) * np.linalg.norm(prediction, axis=1)

    def safe_ratio(a, b):
        return np.divide(a, b, out=np.full_like(a, np.nan), where=b > 0)

    with np.errstate(divide='ignore', invalid='ignore'):
        snr_in = 10 * np.log10(safe_ratio(signal_power, input_power))
        snr_out = 10 * np.log10(safe_ratio(signal_power, numerator))
    frame = metadata[['sample_id', 'patient_id', 'record_id', 'test_artifact', 'r']].copy()
    frame.insert(3, 'model_artifact', model_artifact)
    frame['in_domain'] = model_artifact == test_artifact
    frame['protocol'] = protocol
    frame['K'] = k
    frame['checkpoint_sha256'] = checkpoint_hash
    frame['output_row'] = np.arange(output_start, output_start + len(clean))
    frame['inference_time_ms'] = elapsed_ms / len(clean)
    frame['ssd'] = numerator
    frame['rmse'] = np.sqrt(np.mean(error ** 2, axis=1))
    frame['mae'] = np.mean(np.abs(error), axis=1)
    frame['mad'] = np.max(np.abs(error), axis=1)
    frame['prd_repo'] = 100 * np.sqrt(safe_ratio(numerator, repo_denominator))
    frame['prd_standard'] = 100 * np.sqrt(safe_ratio(numerator, standard_denominator))
    frame['cosine_similarity'] = safe_ratio(np.sum(clean * prediction, axis=1), cosine_denominator)
    frame['snr_in_db'] = snr_in
    frame['snr_out_db'] = snr_out
    frame['snr_improvement_db'] = snr_out - snr_in
    return frame


checkpoint_hashes = {(row['model_artifact'], int(row['protocol'])): row['sha256']
                     for row in CHECKPOINT_INFO}
result_frames = []
test_seed_offset = {'BW': 11, 'EM': 22, 'MA': 33}

for model_artifact in MODEL_ARTIFACTS:
    for test_artifact in TEST_ARTIFACTS:
        for protocol in PROTOCOLS:
            model_key = (model_artifact, protocol)
            test_key = (test_artifact, protocol)
            model = MODELS[model_key]
            prediction_maps = {
                k: np.lib.format.open_memmap(
                    OUT / (f'xhat_model-{model_artifact.lower()}_test-{test_artifact.lower()}'
                           f'_protocol{protocol}_K{k}.npy'),
                    mode='w+', dtype=np.float32, shape=CLEAN.shape,
                ) for k in K_VALUES
            }
            # Common random numbers: for a fixed test artifact/protocol every model
            # starts from the same diffusion RNG state.
            inference_seed = SEED + test_seed_offset[test_artifact] * 100 + protocol
            torch.manual_seed(inference_seed)
            torch.cuda.manual_seed_all(inference_seed)
            description = f'model={model_artifact}, test={test_artifact}, P{protocol}'
            for start in tqdm(range(0, len(CLEAN), BATCH_SIZE), desc=description):
                stop = min(start + BATCH_SIZE, len(CLEAN))
                scale = np.maximum(
                    np.max(np.abs(CLEAN[start:stop]), axis=1, keepdims=True), 1e-8
                ).astype(np.float32)
                condition = torch.from_numpy(
                    (NOISY[test_key][start:stop] / scale)[:, None, :]
                ).to(DEVICE)
                running_sum = torch.zeros_like(condition)
                synchronize()
                tick = time.perf_counter()
                with torch.inference_mode():
                    for shot in range(1, max(K_VALUES) + 1):
                        output = model.denoising(
                            condition, use_ddim=True, ddim_steps=DDIM_STEPS,
                            ddim_eta=DDIM_ETA, num_shots=1,
                        )
                        if output.shape != condition.shape or not torch.isfinite(output).all():
                            raise RuntimeError(f'Invalid output: {description}, shot={shot}')
                        running_sum += output
                        if shot in K_VALUES:
                            synchronize()
                            elapsed_ms = (time.perf_counter() - tick) * 1000
                            prediction = (running_sum / shot).cpu().numpy()[:, 0, :] * scale
                            prediction_maps[shot][start:stop] = prediction
                            result_frames.append(compute_metric_frame(
                                CLEAN[start:stop], NOISY[test_key][start:stop], prediction,
                                MANIFEST[test_key].iloc[start:stop].reset_index(drop=True),
                                model_artifact, test_artifact, protocol, shot,
                                elapsed_ms, start, checkpoint_hashes[model_key],
                            ))
            for values in prediction_maps.values():
                values.flush()

per_sample = pd.concat(result_frames, ignore_index=True)
expected_rows = (len(CLEAN) * len(MODEL_ARTIFACTS) * len(TEST_ARTIFACTS)
                 * len(PROTOCOLS) * len(K_VALUES))
assert len(per_sample) == expected_rows
assert not per_sample.duplicated(
    ['model_artifact', 'test_artifact', 'protocol', 'K', 'sample_id']
).any()
per_sample.to_csv(OUT / 'metrics_per_sample.csv', index=False)
print('Metric rows:', len(per_sample), '| expected:', expected_rows)


## 10. Ma trận tổng quát hóa, OOD gap và biểu đồ


In [ ]:
METRICS = [
    'ssd', 'rmse', 'mae', 'mad', 'prd_repo', 'prd_standard',
    'cosine_similarity', 'snr_in_db', 'snr_out_db',
    'snr_improvement_db', 'inference_time_ms',
]
per_sample['r_group'] = pd.cut(
    per_sample.r,
    bins=[0.2, 0.6, 1.0, 1.5, 2.001],
    right=False,
    labels=['0.20-<0.60', '0.60-<1.00', '1.00-<1.50', '1.50-2.00'],
)
assert per_sample.r_group.notna().all()


def metric_stats(frame, prefix=''):
    result = {}
    for metric in METRICS:
        values = frame[metric].to_numpy(dtype=np.float64)
        finite = values[np.isfinite(values)]
        result[f'{prefix}{metric}_valid_N'] = len(finite)
        result[f'{prefix}{metric}_nonfinite_N'] = len(values) - len(finite)
        result[f'{prefix}{metric}_mean'] = float(np.mean(finite)) if len(finite) else np.nan
        result[f'{prefix}{metric}_std'] = float(np.std(finite, ddof=0)) if len(finite) else np.nan
    return result


summary_rows = []
group_columns = ['model_artifact', 'test_artifact', 'protocol', 'K']
for keys, frame in per_sample.groupby(group_columns):
    model_artifact, test_artifact, protocol, k = keys
    base = {
        'model_artifact': model_artifact,
        'test_artifact': test_artifact,
        'protocol': protocol,
        'K': k,
        'r_group': 'ALL',
        'N': len(frame),
        'in_domain': model_artifact == test_artifact,
    }
    summary_rows.append({**base, **metric_stats(frame)})
    for r_group, part in frame.groupby('r_group', observed=True):
        summary_rows.append({
            **base, 'r_group': str(r_group), 'N': len(part), **metric_stats(part)
        })

# Protocol-pooled rows retain equal protocol weighting because both protocols have
# exactly the same QTDB samples.
for keys, frame in per_sample.groupby(['model_artifact', 'test_artifact', 'K']):
    model_artifact, test_artifact, k = keys
    summary_rows.append({
        'model_artifact': model_artifact,
        'test_artifact': test_artifact,
        'protocol': 'pooled',
        'K': k,
        'r_group': 'ALL',
        'N': len(frame),
        'in_domain': model_artifact == test_artifact,
        **metric_stats(frame),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / 'summary.csv', index=False)

record_rows = []
for keys, frame in per_sample.groupby(
        ['model_artifact', 'test_artifact', 'protocol', 'K', 'record_id']):
    model_artifact, test_artifact, protocol, k, record_id = keys
    record_rows.append({
        'model_artifact': model_artifact,
        'test_artifact': test_artifact,
        'protocol': protocol,
        'K': k,
        'record_id': record_id,
        'N': len(frame),
        **metric_stats(frame),
    })
pd.DataFrame(record_rows).to_csv(OUT / 'summary_by_record.csv', index=False)

DISPLAY_K = max(K_VALUES)
display_frame = per_sample[per_sample.K == DISPLAY_K].copy()

# Exact 3x3 matrices, pooled over the two protocols. These are the primary
# generalization results; rows are training domains and columns are test domains.
matrix_rows = []
for (model_artifact, test_artifact), frame in display_frame.groupby(
        ['model_artifact', 'test_artifact']):
    row = {
        'model_artifact': model_artifact,
        'test_artifact': test_artifact,
        'K': DISPLAY_K,
        'N': len(frame),
        'in_domain': model_artifact == test_artifact,
    }
    for metric in METRICS:
        values = frame[metric].replace([np.inf, -np.inf], np.nan).dropna()
        row[f'{metric}_mean'] = values.mean()
        row[f'{metric}_std'] = values.std(ddof=0)
    matrix_rows.append(row)
cross_noise_matrix = pd.DataFrame(matrix_rows)
cross_noise_matrix.to_csv(OUT / 'cross_noise_matrix.csv', index=False)
display(cross_noise_matrix)

# Model-level OOD report. OOD macro first averages within each unseen artifact,
# then averages the two artifact means so every artifact has equal weight.
generalization_rows = []
for model_artifact in MODEL_ARTIFACTS:
    model_frame = display_frame[display_frame.model_artifact == model_artifact]
    in_domain = model_frame[model_frame.test_artifact == model_artifact]
    ood = model_frame[model_frame.test_artifact != model_artifact]
    ood_by_artifact = ood.groupby('test_artifact')[
        ['snr_improvement_db', 'prd_standard', 'cosine_similarity']
    ].mean()
    row = {
        'model_artifact': model_artifact,
        'K': DISPLAY_K,
        'in_domain_N': len(in_domain),
        'ood_N': len(ood),
        'snr_improvement_in_domain_db': in_domain.snr_improvement_db.mean(),
        'snr_improvement_ood_macro_db': ood_by_artifact.snr_improvement_db.mean(),
        'snr_improvement_ood_worst_db': ood_by_artifact.snr_improvement_db.min(),
        'prd_standard_in_domain_pct': in_domain.prd_standard.mean(),
        'prd_standard_ood_macro_pct': ood_by_artifact.prd_standard.mean(),
        'prd_standard_ood_worst_pct': ood_by_artifact.prd_standard.max(),
        'cosine_in_domain': in_domain.cosine_similarity.mean(),
        'cosine_ood_macro': ood_by_artifact.cosine_similarity.mean(),
        'cosine_ood_worst': ood_by_artifact.cosine_similarity.min(),
    }
    row['snr_generalization_drop_db'] = (
        row['snr_improvement_in_domain_db'] - row['snr_improvement_ood_macro_db']
    )
    row['prd_generalization_increase_pct_point'] = (
        row['prd_standard_ood_macro_pct'] - row['prd_standard_in_domain_pct']
    )
    row['cosine_generalization_drop'] = row['cosine_in_domain'] - row['cosine_ood_macro']
    for test_artifact in TEST_ARTIFACTS:
        part = model_frame[model_frame.test_artifact == test_artifact]
        row[f'snr_improvement_test_{test_artifact}_db'] = part.snr_improvement_db.mean()
    generalization_rows.append(row)

generalization = pd.DataFrame(generalization_rows)
generalization.to_csv(OUT / 'generalization_by_model.csv', index=False)
display(generalization)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#4A4A4A',
    'axes.labelcolor': '#222222',
    'axes.titleweight': 'bold',
    'axes.grid': False,
    'font.size': 10,
})
ORDER = list(MODEL_ARTIFACTS)


def plot_matrix(metric, title, unit, cmap, filename):
    matrix = (cross_noise_matrix.pivot(
        index='model_artifact', columns='test_artifact', values=f'{metric}_mean'
    ).reindex(index=ORDER, columns=list(TEST_ARTIFACTS)))
    values = matrix.to_numpy(dtype=float)
    fig, axis = plt.subplots(figsize=(6.8, 5.5))
    image = axis.imshow(values, cmap=cmap, aspect='equal')
    axis.set(
        xticks=np.arange(len(TEST_ARTIFACTS)),
        xticklabels=TEST_ARTIFACTS,
        yticks=np.arange(len(ORDER)),
        yticklabels=ORDER,
        xlabel='Test noise',
        ylabel='Model training noise',
        title=f'{title} (protocols pooled, K={DISPLAY_K})',
    )
    axis.grid(False)
    threshold = np.nanmedian(values)
    for row_index in range(values.shape[0]):
        for column_index in range(values.shape[1]):
            value = values[row_index, column_index]
            color = 'white' if value < threshold else '#111111'
            axis.text(column_index, row_index, f'{value:.3f}', ha='center',
                      va='center', color=color, fontweight='bold')
    fig.colorbar(image, ax=axis, label=unit)
    fig.tight_layout()
    fig.savefig(OUT / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)


plot_matrix('snr_improvement_db', 'Cross-noise SNR improvement', 'dB',
            'cividis', 'cross_noise_snr_improvement_heatmap.png')
plot_matrix('prd_standard', 'Cross-noise standard PRD', '%',
            'magma_r', 'cross_noise_prd_heatmap.png')
plot_matrix('cosine_similarity', 'Cross-noise cosine similarity', 'score',
            'viridis', 'cross_noise_cosine_heatmap.png')

# In-domain vs OOD macro comparison. Exact gaps remain in the CSV.
positions = np.arange(len(generalization))
width = 0.36
fig, axis = plt.subplots(figsize=(8.2, 5.2))
axis.bar(positions - width / 2, generalization.snr_improvement_in_domain_db,
         width, color='#4C78A8', label='In-domain')
axis.bar(positions + width / 2, generalization.snr_improvement_ood_macro_db,
         width, color='#F2CF5B', edgecolor='#333333', label='OOD macro')
axis.axhline(0, color='#333333', linewidth=1)
axis.set(
    xticks=positions,
    xticklabels=generalization.model_artifact,
    xlabel='Model training noise',
    ylabel='Mean SNR improvement (dB)',
    title=f'In-domain vs unseen-noise generalization (K={DISPLAY_K})',
)
axis.grid(axis='y', alpha=0.22)
axis.legend()
fig.tight_layout()
fig.savefig(OUT / 'in_domain_vs_ood_snr.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

run_info = {
    'model_artifacts': list(MODEL_ARTIFACTS),
    'test_artifacts': list(TEST_ARTIFACTS),
    'protocols': list(PROTOCOLS),
    'test_records': list(selected_records),
    'N_clean': len(CLEAN),
    'sampling_rate': TARGET_FS,
    'segment_length': LENGTH,
    'preprocessing_version': PREPROCESSING_VERSION,
    'model_normalization': MODEL_NORMALIZATION,
    'K': list(K_VALUES),
    'batch_size': BATCH_SIZE,
    'ddim_steps': DDIM_STEPS,
    'ddim_eta': DDIM_ETA,
    'common_random_numbers_across_models': True,
    'checkpoint_info': CHECKPOINT_INFO,
    'device': str(DEVICE),
    'torch_version': torch.__version__,
    'numpy_version': np.__version__,
    'scipy_version': scipy.__version__,
    'wfdb_version': wfdb.__version__,
    'python_version': platform.python_version(),
    'full_evaluation': MAX_TEST_RECORDS is None and MAX_TEST_SEGMENTS is None,
}
(OUT / 'run_info.json').write_text(json.dumps(run_info, indent=2), encoding='utf-8')

archive_path = OUT.parent / 'qtdb_cross_noise_generalization.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(OUT))
print('Saved:', OUT, '| archive:', archive_path)


## Output chính

- `metrics_per_sample.csv`: metric từng segment cho toàn bộ 3×3×2 protocol;
- `summary.csv`: mean/SD theo model, test noise, protocol, K và mức `r`;
- `cross_noise_matrix.csv`: ma trận 3×3 gộp hai protocol;
- `generalization_by_model.csv`: in-domain, OOD macro, OOD worst-case và generalization gap;
- `summary_by_record.csv`: kiểm tra độ ổn định theo QTDB record;
- `checkpoint_manifest.csv`, `test_pair_manifest.csv`, `run_info.json`: provenance;
- ba heatmap SNR/PRD/cosine và biểu đồ in-domain so với OOD;
- `qtdb_cross_noise_generalization.zip`: toàn bộ kết quả.

Giá trị gap dương trong `snr_generalization_drop_db`, `prd_generalization_increase_pct_point` hoặc `cosine_generalization_drop` biểu thị hiệu năng xấu đi khi chuyển sang nhiễu ngoài miền.
